# 03 - Batch Experiments (OpenAI)

This notebook executes flaky test classification experiments using the OpenAI Batch API.

## Workflow
1. Define experiments
2. Validate request files
3. Upload batch request files
4. Create OpenAI Batch jobs
5. Monitor batch execution
6. Download batch results
7. Parse predictions
8. Save predictions for evaluation

This notebook follows the same workflow as `02_batch_experiments.ipynb`, replacing the Gemini Batch API with the OpenAI Batch API.

In [1]:
%pip install openai tqdm

Defaulting to user installation because normal site-packages is not writeable
Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 26.1.1 -> 26.1.2
[notice] To update, run: python.exe -m pip install --upgrade pip


In [ ]:
# ==========================================================
# OpenAI Configuration
# ==========================================================

from pathlib import Path

from openai import OpenAI

from dotenv import load_dotenv
import os

load_dotenv()

OPENAI_API_KEY = os.getenv("OPENAI_API_KEY")

# Initialize client
client = OpenAI( api_key=OPENAI_API_KEY )

# Model Configuration
MODEL_NAME = "gpt-5-mini"
BATCH_ENDPOINT_URL = "/v1/responses"
MAX_OUTPUT_TOKENS = 2048


# ============================================================
# Project Directories
# ============================================================

PROJECT_ROOT = Path.cwd().parent

DATA_DIR = PROJECT_ROOT / "datasets"

REQUEST_DIR = PROJECT_ROOT / "generated_prompts" / "openai_batch_requests"

RESULTS_DIR = PROJECT_ROOT / "results"

REQUEST_DIR.mkdir(parents=True, exist_ok=True)
RESULTS_DIR.mkdir(parents=True, exist_ok=True)

print(f"Model      : {MODEL_NAME}")
print(f"Requests   : {REQUEST_DIR}")
print(f"Results    : {RESULTS_DIR}")

Model      : gpt-5-mini
Requests   : d:\university works\Final-Year_Firts_sem\FYP\INFO\REPO\CA-Classification-Framework\generated_prompts\openai_batch_requests
Results    : d:\university works\Final-Year_Firts_sem\FYP\INFO\REPO\CA-Classification-Framework\results


In [3]:
# ============================================================
# Experiment Definitions
# ============================================================

EXPERIMENTS = [
    {
        "name": "zero_shot_without_context",
        "prompt_type": "zero_shot",
        "context_enabled": False,
    },
    {
        "name": "zero_shot_with_context",
        "prompt_type": "zero_shot",
        "context_enabled": True,
    },
    {
        "name": "zero_shot_cot_without_context",
        "prompt_type": "zero_shot_cot",
        "context_enabled": False,
    },
    {
        "name": "zero_shot_cot_with_context",
        "prompt_type": "zero_shot_cot",
        "context_enabled": True,
    },
    {
        "name": "few_shot_cot_without_context",
        "prompt_type": "few_shot_cot",
        "context_enabled": False,
    },
    {
        "name": "few_shot_cot_with_context",
        "prompt_type": "few_shot_cot",
        "context_enabled": True,
    },
]

print(f"Total Experiments: {len(EXPERIMENTS)}")

for exp in EXPERIMENTS:
    print(f"• {exp['name']}")

Total Experiments: 6
• zero_shot_without_context
• zero_shot_with_context
• zero_shot_cot_without_context
• zero_shot_cot_with_context
• few_shot_cot_without_context
• few_shot_cot_with_context


## 3. Load Evaluation Dataset

Load the evaluation dataset that will be used to generate prompts for every
experiment. A lookup table is also created to enable efficient retrieval of
samples using their unique identifier when predictions are merged with
ground truth later.

In [4]:
# ============================================================
# Imports
# ============================================================

import json
import time

import pandas as pd
from tqdm.auto import tqdm

# ============================================================
# Load Evaluation Dataset
# ============================================================

EVALUATION_DATASET = DATA_DIR / "evaluation_dataset.jsonl"

if not EVALUATION_DATASET.exists():
    raise FileNotFoundError(
        f"Evaluation dataset not found:\n{EVALUATION_DATASET}"
    )

df = pd.read_json(EVALUATION_DATASET, lines=True)

df_lookup = df.set_index("id")

print(f"✓ Loaded {len(df)} evaluation samples.")

df.head()

C:\Users\ASUS\AppData\Roaming\Python\Python314\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


✓ Loaded 2210 evaluation samples.


,id,test_id,isFlaky,issue_category,repo_url,issue_commit,fixed_commit,test_code,helper_methods_json,failure_log,code_under_test_json,test_code_original,helper_methods_json_original,failure_log_original,code_under_test_original,has_helper_methods,has_code_under_test,has_failure_log,context_score
0,857,ormlitecore59309e55,True,Order Dependent,https://github.com/j256/ormlite-core,59309e51c61e8a63cb5fd24a5a7607b668a5f095,c80bde196ca152ecc8a3c4f38f77dbe5a4ea3232,@Test\n\tpublic void testSetObjectCacheThrow()...,{},org.opentest4j.AssertionFailedError: Unexpecte...,{'com.j256.ormlite.dao.BaseDaoImpl': {'setObje...,@Test\n\tpublic void testSetObjectCacheThrow()...,{},Failed Rounds: 10/10\norg.opentest4j.Assertion...,{'com.j256.ormlite.dao.BaseDaoImpl': {'setObje...,False,True,True,3
1,1574,Closure-144-21,False,Non-Flaky,https://github.com/google/closure-compiler,c9e89727dc8063d087d28e42629606f4fd74a6e5,465282f1ca28a208b06c47b55fd292d4631c55da,public void testExportMultiple3() throws Excep...,{'compileAndCheck': 'private void compileAndCh...,junit.framework.ComparisonFailure: expected:<....,{'com.google.javascript.jscomp.Result': {'<ini...,public void testExportMultiple3() throws Excep...,{'compileAndCheck': 'private void compileAndCh...,Failed Rounds: 1/1\njunit.framework.Comparison...,{'com.google.javascript.jscomp.Result': {'<ini...,True,True,True,5
2,1483,Closure-115-5,False,Non-Flaky,https://github.com/google/closure-compiler,2d6e1c78f41248fbbb1eec43b23e7430e2bc7885,4597738e8898f738c1f969fe90479728be81cc80,public void testInlineFunctions6() {\n\n te...,"{'test': 'public void test(String js, String e...",junit.framework.AssertionFailedError:\nExpecte...,{'com.google.javascript.jscomp.DiagnosticType'...,public void testInlineFunctions6() {\n // m...,{'test': '/** * Verifies that the compiler ...,Failed Rounds: 1/1\njunit.framework.AssertionF...,{'com.google.javascript.jscomp.DiagnosticType'...,True,True,True,5
3,431,ignite3modulesstoragerocksdb19c8a82testAbortWrite,True,Implementation Dependent,https://github.com/apache/ignite-3,19c8a824bd9d31f0d0dbd3fbbdd2a32ee360cab2,de6ee0702398f9ce3022a8e265c633857f3a3d88,.\n */\n @Test\n public void testAbo...,{'read': 'protected BinaryRow read(RowId rowId...,org.junit.jupiter.api.extension.ParameterResol...,{'org.apache.ignite.internal.hlc.HybridTimesta...,.\n */\n @Test\n public void testAbo...,{'read': '/** * Reads a row. */ ...,Failed Rounds: 84/101\norg.junit.jupiter.api.e...,{'org.apache.ignite.internal.hlc.HybridTimesta...,True,True,True,5
4,1563,Closure-144-10,False,Non-Flaky,https://github.com/google/closure-compiler,c9e89727dc8063d087d28e42629606f4fd74a6e5,465282f1ca28a208b06c47b55fd292d4631c55da,public void testVariableArgumentsTypesAnnotati...,{'assertTypeAnnotations': 'private void assert...,junit.framework.ComparisonFailure: expected:<....,{'com.google.javascript.rhino.Node': {'getDoub...,public void testVariableArgumentsTypesAnnotati...,{'assertTypeAnnotations': 'private void assert...,Failed Rounds: 1/1\njunit.framework.Comparison...,{'com.google.javascript.rhino.Node': {'getDoub...,True,True,True,5


### Import Prompt Builder

Prompt engineering logic is **reused unchanged** from `utils/prompt_builder.py`,
the same utility used by `classification-phase-01-gemini.ipynb`.

In [5]:
import sys

sys.path.insert(0, str(PROJECT_ROOT))

from utils.prompt_builder import build_prompt

## 4. Generate OpenAI Batch Request Files

For every experiment, this section iterates through the evaluation dataset,
builds the prompt using the existing `build_prompt` utility (prompt
engineering logic is **not** modified here), and writes one OpenAI Batch API
request per sample.

Each line follows the OpenAI Batch API format:

```json
{
  "custom_id": "sample_<id>",
  "method": "POST",
  "url": "/v1/responses",
  "body": {
    "model": "gpt-5-mini",
    "input": "...generated prompt...",
    "max_output_tokens": 2048
  }
}
```

Request files are written to `generated_prompts/openai_batch_requests/`.
If a request file already exists, generation is skipped for that experiment
so the notebook can be re-run without unnecessary rework. Pass
`overwrite=True` to force regeneration.

In [6]:
# ============================================================
# Build OpenAI Batch Request Files
# ============================================================

def build_batch_request_line(sample: dict, prompt_type: str, context_enabled: bool) -> dict:
    """
    Build a single OpenAI Batch API request record for one dataset sample.

    Reuses the existing `build_prompt` utility for prompt engineering.
    """

    prompt = build_prompt(
        sample=sample,
        strategy=prompt_type,
        include_context=context_enabled,
    )

    return {
        "custom_id": f"sample_{sample['id']}",
        "method": "POST",
        "url": BATCH_ENDPOINT_URL,
        "body": {
            "model": MODEL_NAME,
            "input": prompt,
            "max_output_tokens": MAX_OUTPUT_TOKENS,
        },
    }


def generate_request_file(experiment: dict, overwrite: bool = False):
    """
    Generate a JSONL OpenAI Batch request file for one experiment.

    Skips generation if the file already exists, unless overwrite=True.
    """

    request_file = REQUEST_DIR / f"{experiment['name']}.jsonl"

    if request_file.exists() and not overwrite:
        print(f"↻ Skipping generation (exists): {request_file.name}")
        return request_file

    print(f"Generating prompts for: {experiment['name']}")

    with open(request_file, "w", encoding="utf-8") as f:

        for _, row in tqdm(
            df.iterrows(),
            total=len(df),
            desc=experiment["name"],
        ):
            sample = row.to_dict()

            record = build_batch_request_line(
                sample=sample,
                prompt_type=experiment["prompt_type"],
                context_enabled=experiment["context_enabled"],
            )

            f.write(json.dumps(record, ensure_ascii=False) + "\n")

    print(f"✓ Wrote request file: {request_file}")

    return request_file

In [7]:
# ============================================================
# Generate Request Files for All Experiments
# ============================================================

for experiment in EXPERIMENTS:
    experiment["request_file"] = generate_request_file(experiment, overwrite=False)

Generating prompts for: zero_shot_without_context


zero_shot_without_context: 100%|██████████| 2210/2210 [00:02<00:00, 1026.28it/s]


✓ Wrote request file: d:\university works\Final-Year_Firts_sem\FYP\INFO\REPO\CA-Classification-Framework\generated_prompts\openai_batch_requests\zero_shot_without_context.jsonl
Generating prompts for: zero_shot_with_context


zero_shot_with_context: 100%|██████████| 2210/2210 [00:02<00:00, 844.16it/s]


✓ Wrote request file: d:\university works\Final-Year_Firts_sem\FYP\INFO\REPO\CA-Classification-Framework\generated_prompts\openai_batch_requests\zero_shot_with_context.jsonl
Generating prompts for: zero_shot_cot_without_context


...

✓ Wrote request file: d:\university works\Final-Year_Firts_sem\FYP\INFO\REPO\CA-Classification-Framework\generated_prompts\openai_batch_requests\zero_shot_cot_with_context.jsonl
Generating prompts for: few_shot_cot_without_context


few_shot_cot_without_context: 100%|██████████| 2210/2210 [00:02<00:00, 928.41it/s] 


✓ Wrote request file: d:\university works\Final-Year_Firts_sem\FYP\INFO\REPO\CA-Classification-Framework\generated_prompts\openai_batch_requests\few_shot_cot_without_context.jsonl
Generating prompts for: few_shot_cot_with_context


few_shot_cot_with_context: 100%|██████████| 2210/2210 [00:03<00:00, 711.25it/s]

✓ Wrote request file: d:\university works\Final-Year_Firts_sem\FYP\INFO\REPO\CA-Classification-Framework\generated_prompts\openai_batch_requests\few_shot_cot_with_context.jsonl


## 5. Validate Request Files

Before uploading, verify that every request file exists and is not empty.

In [8]:
# ============================================================
# Validate Request Files
# ============================================================

print("Validating experiment request files...\n")

for experiment in EXPERIMENTS:
    request_file = experiment.get("request_file") or (
        REQUEST_DIR / f"{experiment['name']}.jsonl"
    )

    if not request_file.exists():
        raise FileNotFoundError(
            f"Missing request file:\n{request_file}"
        )

    if request_file.stat().st_size == 0:
        raise ValueError(
            f"Request file is empty:\n{request_file}"
        )

    experiment["request_file"] = request_file

    print(f"✓ {experiment['name']}")

print(f"\nAll {len(EXPERIMENTS)} request files are available.")

Validating experiment request files...

✓ zero_shot_without_context
✓ zero_shot_with_context
✓ zero_shot_cot_without_context
✓ zero_shot_cot_with_context
✓ few_shot_cot_without_context
✓ few_shot_cot_with_context

All 6 request files are available.


## 6. Batch Execution Functions

This section implements the reusable functions required to execute each
batch experiment using the OpenAI Batch API:

1. Upload the request JSONL file.
2. Create an OpenAI Batch job.
3. Monitor the job until completion.
4. Download the generated response.
5. Convert the response into the standardized prediction format.
6. Save the prediction results.

These functions are reused for all six experiments to ensure a consistent
execution process.

### 6.1 Upload Request File

The first step in the execution workflow is uploading the generated JSONL
request file to the OpenAI Files API with `purpose="batch"`.

The uploaded file is later referenced when creating the batch job.

In [9]:
# ============================================================
# Upload Request File
# ============================================================

def upload_request_file(request_file: Path):
    """
    Upload a JSONL request file to the OpenAI Files API.

    Args:
        request_file (Path): Path to the JSONL request file.

    Returns:
        Uploaded OpenAI file object.
    """

    print(f"Uploading: {request_file.name}")

    with open(request_file, "rb") as f:
        uploaded_file = client.files.create(
            file=f,
            purpose="batch",
        )

    print(f"✓ Upload completed")
    print(f"File Id : {uploaded_file.id}")

    return uploaded_file

In [10]:
experiment = EXPERIMENTS[0]

uploaded_file = upload_request_file(experiment["request_file"])

Uploading: zero_shot_without_context.jsonl
✓ Upload completed
File Id : file-8BdCPLfn3sQ6W7Wu9d5MQj


### 6.2 Create Batch Job

After uploading the request file, an OpenAI Batch job is created.

The batch job references the uploaded JSONL file and specifies the
`/v1/responses` endpoint that should process every request contained in the
file.

The returned batch object is used to monitor execution and later download
the generated predictions.

In [11]:
# ============================================================
# Create Batch Job
# ============================================================

def create_batch_job(uploaded_file):
    """
    Create an OpenAI Batch job.

    Args:
        uploaded_file: Uploaded OpenAI file object.

    Returns:
        Batch job object.
    """

    print("Creating batch job...")

    batch_job = client.batches.create(
        input_file_id=uploaded_file.id,
        endpoint=BATCH_ENDPOINT_URL,
        completion_window="24h",
    )

    print("✓ Batch job created")
    print(f"Batch Job : {batch_job.id}")
    print(f"State     : {batch_job.status}")

    return batch_job

In [12]:
batch_job = create_batch_job(uploaded_file)

Creating batch job...
✓ Batch job created
Batch Job : batch_6a687aec18a4819084ac7c03dc597357
State     : validating


### 6.3 Monitor Batch Job

After a batch job is created, it executes asynchronously.

This function periodically checks the job status until it reaches a
terminal state.

Possible states include:

- validating
- in_progress
- finalizing
- completed
- failed
- expired
- cancelled

Only completed jobs are downloaded in the next step.

In [44]:
# ============================================================
# Wait for Batch Completion
# ============================================================

TERMINAL_SUCCESS_STATES = {"completed"}
TERMINAL_FAILURE_STATES = {"failed", "expired", "cancelled"}


def _format_batch_errors(batch_job):
    """
    Extract readable validation/runtime errors from a failed OpenAI batch.
    Handles object-like and dict-like SDK shapes.
    """

    errors = getattr(batch_job, "errors", None)
    if errors is None and isinstance(batch_job, dict):
        errors = batch_job.get("errors")

    if not errors:
        return "No error details returned by API."

    # Object shape: errors.data
    data = getattr(errors, "data", None)

    # Dict shape: {"data": [...]}
    if data is None and isinstance(errors, dict):
        data = errors.get("data")

    # Some SDK responses return list directly
    if data is None and isinstance(errors, list):
        data = errors

    if not data:
        return str(errors)

    lines = []
    for item in data:
        if isinstance(item, dict):
            code = item.get("code", "unknown_code")
            message = item.get("message", "")
            param = item.get("param")
        else:
            code = getattr(item, "code", "unknown_code")
            message = getattr(item, "message", "")
            param = getattr(item, "param", None)

        if param:
            lines.append(f"- {code}: {message} (param={param})")
        else:
            lines.append(f"- {code}: {message}")

    return "\n".join(lines) if lines else "No structured error entries returned."


def wait_for_batch_completion(batch_job, poll_interval=30):
    """
    Wait until an OpenAI Batch job completes.

    Args:
        batch_job: OpenAI batch job object.
        poll_interval (int): Seconds between status checks.

    Returns:
        Completed batch job object.
    """

    print("\nWaiting for batch job to complete...\n")

    while True:

        batch_job = client.batches.retrieve(batch_job.id)

        completed = batch_job.request_counts.completed if batch_job.request_counts else "?"
        total = batch_job.request_counts.total if batch_job.request_counts else "?"

        print(f"{time.strftime('%H:%M:%S')} | {batch_job.status} | {completed}/{total}")

        if batch_job.status in TERMINAL_SUCCESS_STATES:
            print("\n✓ Batch job completed successfully.")
            return batch_job

        if batch_job.status in TERMINAL_FAILURE_STATES:
            error_details = _format_batch_errors(batch_job)
            raise RuntimeError(
                f"Batch job ended with status: {batch_job.status}\n"
                f"Batch ID: {batch_job.id}\n"
                f"Details:\n{error_details}"
            )

        time.sleep(poll_interval)

In [45]:
completed_batch = wait_for_batch_completion(batch_job)


Waiting for batch job to complete...

18:15:08 | completed | 2210/2210

✓ Batch job completed successfully.


### 6.4 Download Batch Results

Download the raw output file generated by the completed OpenAI Batch job.

The downloaded file contains one JSON response per request submitted to the
Batch API. These responses are stored without modification so they can be
parsed in the next step. If the batch also produced an error file, it is
downloaded alongside the output file.

In [46]:
# ============================================================
# Download Batch Results
# ============================================================

def get_experiment_directory(experiment):
    """
    Create the output directory for an experiment.
    """

    context_folder = (
        "with_context"
        if experiment["context_enabled"]
        else "without_context"
    )

    experiment_dir = (
        RESULTS_DIR
        / MODEL_NAME
        / experiment["prompt_type"]
        / context_folder
    )

    experiment_dir.mkdir(
        parents=True,
        exist_ok=True,
    )

    return experiment_dir


def download_batch_results(batch_job, experiment):
    """
    Download the raw OpenAI Batch response.

    Args:
        batch_job: Completed OpenAI batch job.
        experiment (dict): Experiment configuration.

    Returns:
        Path: Downloaded raw response file.
    """

    if batch_job.output_file_id is None:
        raise RuntimeError("Batch job has no output file.")

    experiment_dir = get_experiment_directory(experiment)

    raw_response_file = experiment_dir / "raw_response.jsonl"

    print("Downloading batch results...")

    content = client.files.content(batch_job.output_file_id)

    with open(raw_response_file, "wb") as f:
        f.write(content.read())

    print("✓ Download completed")
    print(f"Saved to: {raw_response_file}")

    if batch_job.error_file_id:
        error_file = experiment_dir / "raw_errors.jsonl"
        error_content = client.files.content(batch_job.error_file_id)
        with open(error_file, "wb") as f:
            f.write(error_content.read())
        print(f"⚠ Batch reported per-request errors → {error_file}")

    return raw_response_file

In [47]:
batch_job = wait_for_batch_completion(batch_job)


Waiting for batch job to complete...

18:15:34 | completed | 2210/2210

✓ Batch job completed successfully.


In [48]:
raw_response_file = download_batch_results(
    batch_job,
    experiment
)

raw_response_file

✓ Download completed
Saved to: d:\university works\Final-Year_Firts_sem\FYP\INFO\REPO\CA-Classification-Framework\results\gpt-5-mini\few_shot_cot\with_context\raw_response.jsonl


WindowsPath('d:/university works/Final-Year_Firts_sem/FYP/INFO/REPO/CA-Classification-Framework/results/gpt-5-mini/few_shot_cot/with_context/raw_response.jsonl')

### 6.5 Inspect Raw Batch Response

Before parsing the complete batch output, inspect a single response record
to verify the response structure returned by the OpenAI Batch API.

This validation step helps ensure the parsing logic matches the actual
response format.

In [18]:
# ============================================================
# Inspect Raw Batch Response
# ============================================================

from pprint import pprint

with open(raw_response_file, "r", encoding="utf-8") as f:
    first_record = json.loads(f.readline())

pprint(first_record)

{'custom_id': 'sample_857',
 'error': None,
 'id': 'batch_req_6a687edf34fc81908adb0009c40c8d1d',
 'response': {'body': {'background': False,
                       'billing': {'payer': 'developer'},
                       'completed_at': 1785232543,
                       'created_at': 1785232540,
                       'error': None,
                       'frequency_penalty': 0.0,
                       'id': 'resp_0d4b596545dde303006a687c9ce860819ea34e2e56dd544b92',
                       'incomplete_details': None,
                       'instructions': None,
                       'max_output_tokens': 2048,
                       'max_tool_calls': None,
                       'metadata': {},
                       'model': 'gpt-5-mini-2025-08-07',
                       'moderation': None,
                       'object': 'response',
                       'output': [{'content': [],
                                   'encrypted_content': 'gAAAAABqaHygvjxopr10IawS7IBMIYXY64wI2ckUwy

### 6.6 Parse Batch Responses

Parse the raw OpenAI Batch responses into a standardized intermediate
format.

Each response contains the unique sample identifier (`custom_id`) together
with the model's JSON prediction. This step extracts those predictions while
preserving the sample identifier, enabling them to be merged with the
evaluation dataset in the next step.

Malformed responses (invalid JSON, missing fields, or per-request API
errors) do not stop execution — they are collected and written to a
`malformed_responses.json` file inside the experiment's results folder.

In [19]:
# ============================================================
# Parse Batch Responses
# ============================================================

def extract_response_text(record):
    """
    Extract the model's text output from a single OpenAI Batch response
    record (Responses API shape).
    """

    body = record["response"]["body"]

    text_parts = []

    for item in body.get("output", []):
        if item.get("type") == "message":
            for content_piece in item.get("content", []):
                if content_piece.get("type") in ("output_text", "text"):
                    text_parts.append(content_piece.get("text", ""))

    return "".join(text_parts).strip()


def parse_batch_results(raw_response_file, experiment):
    """
    Parse OpenAI Batch responses into an intermediate format.

    Args:
        raw_response_file (Path): Downloaded OpenAI Batch response file.
        experiment (dict): Experiment configuration.

    Returns:
        list: Parsed predictions.
    """

    parsed_predictions = []
    failed_predictions = []

    with open(raw_response_file, "r", encoding="utf-8") as f:

        for line_no, line in enumerate(f, start=1):

            record = json.loads(line)

            custom_id = record.get("custom_id")

            try:

                sample_id = int(str(custom_id).replace("sample_", ""))

                if record.get("error"):
                    raise ValueError(f"Batch reported error: {record['error']}")

                response_text = extract_response_text(record)

                # Remove markdown fences if present
                if response_text.startswith("```"):
                    response_text = (
                        response_text.replace("```json", "")
                                      .replace("```", "")
                                      .strip()
                    )

                prediction = json.loads(response_text)

                parsed_predictions.append({
                    "id": sample_id,
                    "prediction": prediction
                })

            except (json.JSONDecodeError, ValueError, KeyError, IndexError, TypeError) as e:

                print("=" * 80)
                print(f"JSON parsing failed")
                print(f"Record : {line_no}")
                print(f"Sample : {custom_id}")
                print(f"Error  : {e}")
                print("=" * 80)

                failed_predictions.append({
                    "line": line_no,
                    "sample_id": custom_id,
                    "error": str(e),
                    "raw_response": line,
                })

                # Skip this record and continue
                continue

    print(f"\n✓ Parsed {len(parsed_predictions)} predictions.")

    if failed_predictions:
        print(f"⚠ Skipped {len(failed_predictions)} malformed responses.")

        experiment_dir = get_experiment_directory(experiment)
        failed_file = experiment_dir / "malformed_responses.json"

        with open(failed_file, "w", encoding="utf-8") as fp:
            json.dump(failed_predictions, fp, indent=2)

        print(f"Failed responses saved to {failed_file}")

    return parsed_predictions

In [20]:
parsed_predictions = parse_batch_results(
    raw_response_file,
    experiment
)

parsed_predictions[:2]


✓ Parsed 2210 predictions.


[{'id': 857,
  'prediction': {'classification': 'Non-Flaky',
   'category': 'Non-Flaky',
   'reasoning': 'The test uses a mock Dao and sets an explicit expectation that dao.setObjectCache(false) will throw a SQLException, then verifies that RuntimeExceptionDao translates that into a RuntimeException. All behavior is driven by the mock interactions defined in the test code, with no timing, ordering, external state, or non-idempotent operations visible in the provided artifact. Therefore there is no evidence of flakiness.',
   'evidence': ['Test Code']}},
 {'id': 1574,
  'prediction': {'classification': 'Non-Flaky',
   'category': 'Non-Flaky',
   'reasoning': 'The test provides fixed input and expected output strings to compileAndCheck and contains no sources of nondeterminism: no randomness, timing, asynchronous operations, external resources, or shared mutable state. The single provided artifact (the Test Code) shows a deterministic unit test, so there is insufficient evidence of flaki

### 6.7 Merge Predictions with Evaluation Dataset

Merge the parsed model predictions with the evaluation dataset.

This step combines the prediction generated by the model with the
corresponding ground truth labels and metadata using the sample identifier
(`id`).

In [21]:
# ============================================================
# Merge Predictions with Evaluation Dataset
# ============================================================

def merge_predictions(parsed_predictions, df_lookup, experiment):
    """
    Merge parsed OpenAI predictions with the evaluation dataset.
    """

    merged_predictions = []
    skipped = []

    for item in parsed_predictions:

        sample_id = item["id"]
        prediction = item["prediction"]

        # Verify sample exists
        if sample_id not in df_lookup.index:
            print(f"⚠ Sample {sample_id} not found in evaluation dataset.")
            skipped.append(sample_id)
            continue

        sample = df_lookup.loc[sample_id]

        merged_predictions.append({

            "id": sample_id,

            "test_id": sample["test_id"],

            "ground_truth_classification": (
                "Flaky"
                if sample["isFlaky"]
                else "Non-Flaky"
            ),

            "ground_truth_category": sample["issue_category"],

            "predicted_classification": prediction.get("classification"),

            "predicted_category": prediction.get("category"),

            "reasoning": prediction.get("reasoning"),

            "evidence": prediction.get("evidence", []),

            "model": MODEL_NAME,

            "prompt_type": experiment["prompt_type"],

            "context_enabled": experiment["context_enabled"],

            "latency_ms": None

        })

    print(f"✓ Merged {len(merged_predictions)} predictions.")

    if skipped:
        print(f"⚠ Skipped {len(skipped)} predictions (missing sample IDs).")

    return merged_predictions

In [22]:
merged_predictions = merge_predictions(
    parsed_predictions,
    df_lookup,
    experiment
)

merged_predictions[:2]

✓ Merged 2210 predictions.


[{'id': 857,
  'test_id': 'ormlitecore59309e55',
  'ground_truth_classification': 'Flaky',
  'ground_truth_category': 'Order Dependent',
  'predicted_classification': 'Non-Flaky',
  'predicted_category': 'Non-Flaky',
  'reasoning': 'The test uses a mock Dao and sets an explicit expectation that dao.setObjectCache(false) will throw a SQLException, then verifies that RuntimeExceptionDao translates that into a RuntimeException. All behavior is driven by the mock interactions defined in the test code, with no timing, ordering, external state, or non-idempotent operations visible in the provided artifact. Therefore there is no evidence of flakiness.',
  'evidence': ['Test Code'],
  'model': 'gpt-5-mini',
  'prompt_type': 'zero_shot',
  'context_enabled': False,
  'latency_ms': None},
 {'id': 1574,
  'test_id': 'Closure-144-21',
  'ground_truth_classification': 'Non-Flaky',
  'ground_truth_category': 'Non-Flaky',
  'predicted_classification': 'Non-Flaky',
  'predicted_category': 'Non-Flaky',

### 6.8 Save Predictions

Save the merged prediction records as a JSON Lines (`.jsonl`) file.

The resulting file serves as the standardized input for the evaluation
notebook, where prediction performance is measured using classification
metrics.

In [23]:
# ============================================================
# Save Predictions
# ============================================================

def save_predictions(merged_predictions, experiment):
    """
    Save merged predictions as a JSONL file.

    Args:
        merged_predictions (list): Final prediction records.
        experiment (dict): Experiment configuration.

    Returns:
        Path: Saved prediction file.
    """

    experiment_dir = get_experiment_directory(experiment)

    prediction_file = experiment_dir / "predictions.jsonl"

    with open(prediction_file, "w", encoding="utf-8") as f:

        for prediction in merged_predictions:

            f.write(
                json.dumps(
                    prediction,
                    ensure_ascii=False
                )
            )

            f.write("\n")

    print(f"✓ Saved {len(merged_predictions)} predictions.")
    print(f"Location: {prediction_file}")

    return prediction_file

In [24]:
prediction_file = save_predictions(
    merged_predictions,
    experiment
)

prediction_file

✓ Saved 2210 predictions.
Location: d:\university works\Final-Year_Firts_sem\FYP\INFO\REPO\CA-Classification-Framework\results\gpt-5-mini\zero_shot\without_context\predictions.jsonl


WindowsPath('d:/university works/Final-Year_Firts_sem/FYP/INFO/REPO/CA-Classification-Framework/results/gpt-5-mini/zero_shot/without_context/predictions.jsonl')

### 6.9 Run Experiment (End-to-End)

Combine every step above into a single reusable function so each experiment
can be executed with one call.

In [25]:
# ============================================================
# Run Experiment
# ============================================================

def run_experiment(experiment):

    print("=" * 80)
    print(f"Running: {experiment['name']}")
    print("=" * 80)

    uploaded_file = upload_request_file(
        experiment["request_file"]
    )

    batch_job = create_batch_job(
        uploaded_file
    )

    batch_job = wait_for_batch_completion(
        batch_job
    )

    raw_response_file = download_batch_results(
        batch_job,
        experiment
    )

    parsed_predictions = parse_batch_results(
        raw_response_file,
        experiment
    )

    merged_predictions = merge_predictions(
        parsed_predictions,
        df_lookup,
        experiment
    )

    prediction_file = save_predictions(
        merged_predictions,
        experiment
    )

    print("\n✓ Experiment completed successfully.")

    return prediction_file

### 6.10 Run the Without-Context Experiments

The `without_context` experiments generate smaller request files, so they
are executed directly with `run_experiment`. The larger `with_context`
experiments are executed later using the chunked runner in Section 7.

In [26]:
WITHOUT_CONTEXT_EXPERIMENTS = [
    exp for exp in EXPERIMENTS if not exp["context_enabled"]
]

print(f"Without-context experiments to run: {len(WITHOUT_CONTEXT_EXPERIMENTS)}")
for exp in WITHOUT_CONTEXT_EXPERIMENTS:
    print(f"  • {exp['name']}")

Without-context experiments to run: 3
  • zero_shot_without_context
  • zero_shot_cot_without_context
  • few_shot_cot_without_context


In [27]:
for experiment in WITHOUT_CONTEXT_EXPERIMENTS:
    prediction_file = run_experiment(experiment)

Running: zero_shot_without_context
Uploading: zero_shot_without_context.jsonl
✓ Upload completed
File Id : file-T51KJgu41Gc5PqeSmiRxah
Creating batch job...
✓ Batch job created
Batch Job : batch_6a688b98f1248190b4f331a9bd6a63d4
State     : validating

Waiting for batch job to complete...

16:29:35 | validating | 0/0
16:30:06 | validating | 0/0
16:30:37 | in_progress | 0/2210
16:31:07 | in_progress | 0/2210
16:31:38 | in_progress | 0/2210
16:32:09 | in_progress | 0/2210
16:32:39 | in_progress | 0/2210
16:33:10 | in_progress | 0/2210
16:33:40 | in_progress | 0/2210
16:34:11 | in_progress | 0/2210
16:34:41 | in_progress | 551/2210
16:35:12 | in_progress | 1068/2210
16:35:43 | in_progress | 1931/2210
16:36:13 | finalizing | 2210/2210
16:36:44 | finalizing | 2210/2210
16:37:14 | finalizing | 2210/2210
16:37:45 | finalizing | 2210/2210
16:38:15 | finalizing | 2210/2210
16:38:46 | finalizing | 2210/2210
16:39:16 | completed | 2210/2210

✓ Batch job completed successfully.
✓ Download completed

In [28]:
# List your current batch jobs
for batch in client.batches.list(limit=20):
    print(f"ID: {batch.id}, Status: {batch.status}")

ID: batch_6a688ffd82948190855707a685d931a3, Status: completed
ID: batch_6a688de91ac481908e8796efcda8a5a1, Status: completed
ID: batch_6a688b98f1248190b4f331a9bd6a63d4, Status: completed
ID: batch_6a687aec18a4819084ac7c03dc597357, Status: completed


## 7. Run With-Context Experiments (Chunked)

`with_context` request files are considerably larger than `without_context`
ones because they include helper methods, failure logs, and production code
alongside the test code. Large files risk exceeding OpenAI Batch API limits
(request-count and file-size limits) and take longer to process as a single
job.

This section adds a **chunked runner** used for the three `with_context`
experiments:

1. Split each large request file into smaller JSONL chunks.
2. Run each chunk through the existing upload → create → wait → download →
   parse pipeline, **one chunk at a time**.
3. Combine the parsed predictions from all chunks for an experiment.
4. Reuse the existing `merge_predictions` and `save_predictions` functions
   unchanged, so the final `predictions.jsonl` file is written in the same
   format and location as the `without_context` experiments.

In [49]:
# ============================================================
# Split a Large Request File into Chunks
# ============================================================

def split_request_file(
    request_file: Path,
    max_requests_per_chunk: int = 500,
    max_chunk_bytes: int = 45_000_000,
):
    """
    Split a JSONL batch request file into smaller chunk files.

    Splits by both request count and approximate file bytes to avoid
    OpenAI Batch validation failures on very large with-context prompts.

    Args:
        request_file (Path): Path to the original JSONL request file.
        max_requests_per_chunk (int): Maximum number of requests per chunk.
        max_chunk_bytes (int): Approximate max UTF-8 bytes per chunk file.

    Returns:
        list[Path]: Paths to the generated chunk files.
    """

    lines = request_file.read_text(encoding="utf-8").splitlines()

    chunks = []
    current_chunk = []
    current_bytes = 0

    for line in lines:
        line_bytes = len((line + "\n").encode("utf-8"))

        would_exceed_count = len(current_chunk) >= max_requests_per_chunk
        would_exceed_bytes = current_bytes + line_bytes > max_chunk_bytes

        if current_chunk and (would_exceed_count or would_exceed_bytes):
            chunks.append(current_chunk)
            current_chunk = []
            current_bytes = 0

        current_chunk.append(line)
        current_bytes += line_bytes

    if current_chunk:
        chunks.append(current_chunk)

    chunk_paths = []

    for idx, chunk_lines in enumerate(chunks, start=1):
        chunk_path = request_file.with_name(
            f"{request_file.stem}_chunk{idx}.jsonl"
        )
        chunk_content = "\n".join(chunk_lines) + "\n"
        chunk_path.write_text(chunk_content, encoding="utf-8")
        chunk_paths.append(chunk_path)

    print(f"Split {request_file.name} into {len(chunk_paths)} chunk(s):")
    for p in chunk_paths:
        size_mb = p.stat().st_size / (1024 * 1024)
        print(f"  • {p.name} ({size_mb:.2f} MB)")

    return chunk_paths

### 7.1 Download Helper for Chunked Jobs

The existing `download_batch_results` function always writes to a fixed
`raw_response.jsonl` filename inside the experiment's results folder.
Running several chunks for the same experiment would overwrite that file
each time, so this helper writes each chunk's raw response to its own
numbered file instead.

In [50]:
# ============================================================
# Download Batch Results for a Chunk
# ============================================================

def download_batch_results_chunk(batch_job, experiment, chunk_index):
    """
    Download a completed batch job's results to a chunk-specific file,
    so multiple chunks for the same experiment don't overwrite each other.
    """

    experiment_dir = get_experiment_directory(experiment)

    raw_response_file = experiment_dir / f"raw_response_chunk{chunk_index}.jsonl"

    print(f"Downloading batch results (chunk {chunk_index})...")

    content = client.files.content(batch_job.output_file_id)

    with open(raw_response_file, "wb") as f:
        f.write(content.read())

    print(f"✓ Download completed")
    print(f"Saved to: {raw_response_file}")

    return raw_response_file

### 7.2 Combine Chunk Raw Responses into a Single File

After all chunks for an experiment finish, this concatenates their
individual `raw_response_chunkN.jsonl` files into one `raw_response.jsonl`,
matching the single-file output the `without_context` experiments already
produce.

In [51]:
# ============================================================
# Combine Chunk Raw Responses
# ============================================================

def combine_chunk_raw_responses(experiment, num_chunks=None, raw_files=None):
    """
    Concatenate raw response files into a single raw_response.jsonl.

    Supports two modes:
    1) num_chunks: reads raw_response_chunk{n}.jsonl files (legacy behavior)
    2) raw_files: explicit list of raw response file paths (recommended)
    """

    experiment_dir = get_experiment_directory(experiment)
    combined_file = experiment_dir / "raw_response.jsonl"

    if raw_files is None:
        if num_chunks is None:
            raise ValueError("Provide either num_chunks or raw_files.")
        raw_files = [
            experiment_dir / f"raw_response_chunk{chunk_index}.jsonl"
            for chunk_index in range(1, num_chunks + 1)
        ]

    with open(combined_file, "w", encoding="utf-8") as out_f:
        for raw_file in raw_files:
            with open(raw_file, "r", encoding="utf-8") as in_f:
                out_f.write(in_f.read())

    print(f"✓ Combined {len(raw_files)} chunk file(s) into: {combined_file}")

    return combined_file

### 7.3 Run a With-Context Experiment in Chunks

This function mirrors `run_experiment`, but processes the request file as a
series of smaller batch jobs instead of one large job, combines their raw
responses into a single file, then merges and saves, so the final output
(`raw_response.jsonl` and `predictions.jsonl`) is identical in structure and
naming to the `without_context` experiments.

In [52]:
# ============================================================
# Run a With-Context Experiment (Chunked)
# ============================================================

def run_experiment_with_context(
    experiment,
    max_requests_per_chunk: int = 500,
    max_chunk_bytes: int = 45_000_000,
    min_requests_per_chunk: int = 1,
):

    print("=" * 80)
    print(f"Running (chunked): {experiment['name']}")
    print("=" * 80)

    chunk_files = split_request_file(
        experiment["request_file"],
        max_requests_per_chunk=max_requests_per_chunk,
        max_chunk_bytes=max_chunk_bytes,
    )

    experiment_dir = get_experiment_directory(experiment)
    recovery_log = experiment_dir / "chunk_recovery_log.jsonl"
    successful_raw_files = []

    def _log_recovery_event(payload):
        with open(recovery_log, "a", encoding="utf-8") as fp:
            fp.write(json.dumps(payload, ensure_ascii=False) + "\n")

    def _run_chunk_with_fallback(chunk_path: Path, label: str):
        """
        Run one chunk. If it fails, recursively split into smaller chunks
        until success or single-request failure.
        """

        print("\n" + "-" * 80)
        print(f"Chunk {label}: {chunk_path.name}")
        print("-" * 80)

        try:
            uploaded_file = upload_request_file(chunk_path)
            batch_job = create_batch_job(uploaded_file)
            batch_job = wait_for_batch_completion(batch_job)

            raw_file = experiment_dir / f"raw_response_{label}.jsonl"
            print(f"Downloading batch results ({label})...")
            content = client.files.content(batch_job.output_file_id)
            with open(raw_file, "wb") as f:
                f.write(content.read())

            print(f"✓ Download completed: {raw_file}")
            successful_raw_files.append(raw_file)
            return

        except RuntimeError as error:
            lines = chunk_path.read_text(encoding="utf-8").splitlines()
            n = len(lines)

            _log_recovery_event({
                "experiment": experiment["name"],
                "chunk_label": label,
                "chunk_file": str(chunk_path),
                "requests": n,
                "error": str(error),
            })

            print("\n✗ Chunk failed.")
            print(f"Experiment : {experiment['name']}")
            print(f"Chunk file : {chunk_path}")
            print(f"Requests   : {n}")
            print(error)

            if n <= min_requests_per_chunk:
                # Isolate and skip only this single problematic request.
                skipped_file = experiment_dir / "skipped_requests.jsonl"
                with open(skipped_file, "a", encoding="utf-8") as sf:
                    sf.write(lines[0] + "\n")

                print("⚠ Skipped one irrecoverable request after recursive split.")
                print(f"Logged in: {skipped_file}")
                return

            mid = n // 2
            left_lines = lines[:mid]
            right_lines = lines[mid:]

            left_path = chunk_path.with_name(f"{chunk_path.stem}_{label}a.jsonl")
            right_path = chunk_path.with_name(f"{chunk_path.stem}_{label}b.jsonl")

            left_path.write_text("\n".join(left_lines) + "\n", encoding="utf-8")
            right_path.write_text("\n".join(right_lines) + "\n", encoding="utf-8")

            print(
                f"↻ Retrying by splitting {label} into {label}a ({len(left_lines)} req) "
                f"and {label}b ({len(right_lines)} req)."
            )

            _run_chunk_with_fallback(left_path, f"{label}a")
            _run_chunk_with_fallback(right_path, f"{label}b")

    for chunk_index, chunk_file in enumerate(chunk_files, start=1):
        _run_chunk_with_fallback(chunk_file, f"chunk{chunk_index}")

    if not successful_raw_files:
        raise RuntimeError(
            "All chunks failed; no successful raw responses were produced. "
            "Check chunk_recovery_log.jsonl for details."
        )

    print(f"\n✓ Completed with {len(successful_raw_files)} successful chunk output file(s).")

    combined_raw_file = combine_chunk_raw_responses(
        experiment,
        raw_files=successful_raw_files,
    )

    all_parsed_predictions = parse_batch_results(combined_raw_file, experiment)

    print(f"Total parsed predictions: {len(all_parsed_predictions)}")

    merged_predictions = merge_predictions(
        all_parsed_predictions,
        df_lookup,
        experiment
    )

    prediction_file = save_predictions(
        merged_predictions,
        experiment
    )

    print("\n✓ With-context experiment completed successfully.")

    return prediction_file

### 7.4 Run the Three With-Context Experiments

Only the `context_enabled = True` experiments are run here, since the
`without_context` experiments already completed in Section 6. Each
experiment is run one at a time, with a short buffer pause between them.

In [53]:
WITH_CONTEXT_EXPERIMENTS = [
    exp for exp in EXPERIMENTS if exp["context_enabled"]
]

print(f"With-context experiments to run: {len(WITH_CONTEXT_EXPERIMENTS)}")
for exp in WITH_CONTEXT_EXPERIMENTS:
    print(f"  • {exp['name']}")

With-context experiments to run: 3
  • zero_shot_with_context
  • zero_shot_cot_with_context
  • few_shot_cot_with_context


In [54]:
for experiment in WITH_CONTEXT_EXPERIMENTS:

    try:
        prediction_file = run_experiment_with_context(
            experiment,
            max_requests_per_chunk=500,
            max_chunk_bytes=45_000_000,
            min_requests_per_chunk=1,
        )
        print(f"✓ Completed: {experiment['name']} -> {prediction_file}")
    except RuntimeError as error:
        print("\n" + "=" * 80)
        print(f"✗ Failed experiment: {experiment['name']}")
        print(error)
        print("=" * 80)

    # Buffer pause before the next experiment.
    print("Waiting 60s before starting the next experiment...")
    time.sleep(60)

Running (chunked): zero_shot_with_context
Split zero_shot_with_context.jsonl into 5 chunk(s):
  • zero_shot_with_context_chunk1.jsonl (19.21 MB)
  • zero_shot_with_context_chunk2.jsonl (18.17 MB)
  • zero_shot_with_context_chunk3.jsonl (17.32 MB)
  • zero_shot_with_context_chunk4.jsonl (19.89 MB)
  • zero_shot_with_context_chunk5.jsonl (6.72 MB)

--------------------------------------------------------------------------------
Chunk chunk1: zero_shot_with_context_chunk1.jsonl
--------------------------------------------------------------------------------
Uploading: zero_shot_with_context_chunk1.jsonl
✓ Upload completed
File Id : file-2JF2v2vd1kbaHdGcuYApmE
Creating batch job...
✓ Batch job created
Batch Job : batch_6a68a4c4b3a48190a6c8def84a6ee02f
State     : validating

Waiting for batch job to complete...

18:17:01 | validating | 0/0
18:17:31 | in_progress | 0/500
18:18:02 | in_progress | 0/500
18:18:33 | in_progress | 11/500
18:19:03 | in_progress | 496/500
18:19:34 | finalizing | 5

## 8. Recover Malformed Batch Responses

This section retries only records whose original model response was not
valid JSON. It never overwrites `raw_response.jsonl` or `predictions.jsonl`.
Recovery artifacts are written under each experiment's `recovery/` folder,
and a validated combined file is saved as `predictions_recovered.jsonl`.

Run the preparation cell first, then run `run_all_recoveries()` when ready
to submit the small retry batches.

In [55]:
# ============================================================
# Safe recovery of malformed model responses
# ============================================================

RECOVERY_EXPERIMENTS = EXPERIMENTS


def get_recovery_directory(experiment):
    recovery_dir = get_experiment_directory(experiment) / 'recovery'
    recovery_dir.mkdir(parents=True, exist_ok=True)
    return recovery_dir


def get_invalid_response_ids(raw_response_file: Path):
    """Return custom_ids whose OpenAI response text is not a JSON object."""
    invalid_ids = []
    with open(raw_response_file, encoding='utf-8') as f:
        for line_no, line in enumerate(f, start=1):
            record = json.loads(line)
            custom_id = record.get('custom_id')
            try:
                if record.get('error'):
                    raise ValueError(f"Batch reported error: {record['error']}")
                response_text = extract_response_text(record)
                if response_text.startswith('```'):
                    response_text = response_text.replace('```json', '').replace('```', '').strip()
                prediction = json.loads(response_text)
                if not isinstance(prediction, dict):
                    raise ValueError('Prediction is not a JSON object')
            except (KeyError, IndexError, TypeError, ValueError, json.JSONDecodeError) as error:
                invalid_ids.append(custom_id)
                print(f'Invalid response | line={line_no} | id={custom_id} | {error}')
    return invalid_ids


def create_retry_request_file(experiment, overwrite=False):
    """Copy only failed original requests into a new recovery JSONL file."""
    experiment_dir = get_experiment_directory(experiment)
    recovery_dir = get_recovery_directory(experiment)
    retry_request_file = recovery_dir / 'retry_requests.jsonl'
    manifest_file = recovery_dir / 'retry_manifest.json'

    if retry_request_file.exists() and not overwrite:
        if not manifest_file.exists():
            raise FileExistsError(
                f'{retry_request_file} exists but its manifest is missing; refusing to reuse it.'
            )
        with open(manifest_file, encoding='utf-8') as f:
            manifest = json.load(f)
        retry_ids = manifest['sample_ids']
        print(f'{experiment["name"]}: reusing {len(retry_ids)} prepared retry request(s)')
        return retry_request_file, retry_ids

    invalid_ids = get_invalid_response_ids(experiment_dir / 'raw_response.jsonl')
    invalid_id_set = set(invalid_ids)

    original_requests = {}
    with open(experiment['request_file'], encoding='utf-8') as f:
        for line in f:
            request = json.loads(line)
            key = request['custom_id']
            if key in invalid_id_set:
                original_requests[key] = line.rstrip('\n')

    missing_requests = invalid_id_set - set(original_requests)
    if missing_requests:
        raise RuntimeError(f'Failed IDs not found in request file: {sorted(missing_requests)}')

    with open(retry_request_file, 'w', encoding='utf-8') as f:
        for sample_id in invalid_ids:
            f.write(original_requests[sample_id] + '\n')

    manifest = {
        'experiment': experiment['name'],
        'source_raw_response': str(experiment_dir / 'raw_response.jsonl'),
        'retry_request_file': str(retry_request_file),
        'sample_ids': invalid_ids,
    }
    with open(manifest_file, 'w', encoding='utf-8') as f:
        json.dump(manifest, f, indent=2)

    print(f'{experiment["name"]}: prepared {len(invalid_ids)} retry request(s)')
    return retry_request_file, invalid_ids


def download_retry_results(batch_job, experiment, overwrite=False):
    """Download retry output without touching the original raw response file."""
    retry_raw_file = get_recovery_directory(experiment) / 'raw_response_retry.jsonl'
    if retry_raw_file.exists() and not overwrite:
        raise FileExistsError(f'{retry_raw_file} already exists; refusing to overwrite it.')
    if batch_job.output_file_id is None:
        raise RuntimeError('Retry batch job has no output file.')

    content = client.files.content(batch_job.output_file_id)
    with open(retry_raw_file, 'wb') as f:
        f.write(content.read())
    return retry_raw_file


def recover_evaluation_fields(response_text):
    """Safely recover only evaluation fields from malformed model JSON."""
    import re

    def extract_string(field_name):
        match = re.search(rf'\"{field_name}\"\s*:\s*\"([^\"]*)\"', response_text)
        return match.group(1) if match else None

    classification = extract_string('classification')
    category = extract_string('category')
    reasoning_match = re.search(
        r'\"reasoning\"\s*:\s*\"(.*?)\"\s*,\s*\"evidence\"\s*:',
        response_text,
        flags=re.DOTALL,
    )
    reasoning = reasoning_match.group(1) if reasoning_match else response_text

    allowed_categories = {
        'Implementation Dependent', 'Order Dependent', 'Non-Idempotent',
        'Time Dependent', 'Non-Flaky',
    }
    if classification not in {'Flaky', 'Non-Flaky'}:
        raise ValueError(f'Invalid recovered classification: {classification!r}')
    if category not in allowed_categories:
        raise ValueError(f'Invalid recovered category: {category!r}')
    if classification == 'Non-Flaky' and category != 'Non-Flaky':
        raise ValueError('Non-Flaky classification must use Non-Flaky category.')
    if classification == 'Flaky' and category == 'Non-Flaky':
        raise ValueError('Flaky classification cannot use Non-Flaky category.')

    return {
        'classification': classification,
        'category': category,
        'reasoning': reasoning,
        'evidence': [],
    }


def parse_retry_results(retry_raw_file, expected_ids, experiment):
    """Parse retries, retaining validated evaluation fields if only JSON formatting failed."""
    parsed = []
    failures = []
    field_recoveries = []
    expected_id_set = set(expected_ids)
    with open(retry_raw_file, encoding='utf-8') as f:
        for line_no, line in enumerate(f, start=1):
            record = json.loads(line)
            custom_id = record.get('custom_id')
            if custom_id not in expected_id_set:
                continue
            sample_id = int(str(custom_id).replace('sample_', ''))
            try:
                if record.get('error'):
                    raise ValueError(f"Batch reported error: {record['error']}")
                response_text = extract_response_text(record)
                if response_text.startswith('```'):
                    response_text = response_text.replace('```json', '').replace('```', '').strip()
                prediction = json.loads(response_text)
                if not isinstance(prediction, dict):
                    raise ValueError('Prediction is not a JSON object')
                parsed.append({'id': sample_id, 'prediction': prediction})
            except json.JSONDecodeError as error:
                try:
                    prediction = recover_evaluation_fields(response_text)
                    parsed.append({'id': sample_id, 'prediction': prediction})
                    field_recoveries.append({
                        'line': line_no, 'sample_id': sample_id, 'json_error': str(error),
                    })
                except ValueError as recovery_error:
                    failures.append({
                        'line': line_no, 'sample_id': sample_id,
                        'error': f'{error}; field recovery failed: {recovery_error}',
                    })
            except (KeyError, IndexError, TypeError, ValueError) as error:
                failures.append({'line': line_no, 'sample_id': sample_id, 'error': str(error)})

    if field_recoveries:
        audit_file = get_recovery_directory(experiment) / 'retry_field_recoveries.json'
        with open(audit_file, 'w', encoding='utf-8') as f:
            json.dump(field_recoveries, f, indent=2)
        print(f'⚠ Recovered evaluation fields from {len(field_recoveries)} malformed JSON response(s): {audit_file}')

    if failures:
        failure_file = get_recovery_directory(experiment) / 'retry_parse_failures.json'
        with open(failure_file, 'w', encoding='utf-8') as f:
            json.dump(failures, f, indent=2)
        raise RuntimeError(f'Retry produced {len(failures)} malformed response(s); details: {failure_file}')

    parsed_numeric_ids = {item['id'] for item in parsed}
    expected_numeric_ids = {int(str(sid).replace('sample_', '')) for sid in expected_id_set}

    if len(parsed) != len(parsed_numeric_ids):
        raise RuntimeError('Retry output contains duplicate sample IDs.')
    if parsed_numeric_ids != expected_numeric_ids:
        raise RuntimeError(
            f'Retry validation failed. Missing={sorted(expected_numeric_ids - parsed_numeric_ids)}; '
            f'Unexpected={sorted(parsed_numeric_ids - expected_numeric_ids)}'
        )
    return parsed


def write_recovered_predictions(experiment, retry_predictions, expected_ids, overwrite=False):
    """Create a new complete prediction file; leave predictions.jsonl unchanged."""
    experiment_dir = get_experiment_directory(experiment)
    original_file = experiment_dir / 'predictions.jsonl'
    recovered_file = experiment_dir / 'predictions_recovered.jsonl'

    if recovered_file.exists() and not overwrite:
        raise FileExistsError(f'{recovered_file} already exists; refusing to overwrite it.')

    with open(original_file, encoding='utf-8') as f:
        original_predictions = [json.loads(line) for line in f]

    original_ids = {str(item['id']) for item in original_predictions}
    retry_ids = {str(item['id']) for item in retry_predictions}
    expected_numeric_ids = {str(int(str(sid).replace('sample_', ''))) for sid in expected_ids}

    if original_ids & retry_ids:
        raise RuntimeError('Refusing to merge: a retry ID already exists in predictions.jsonl.')
    if retry_ids != expected_numeric_ids:
        raise RuntimeError('Refusing to merge: retry prediction IDs do not match expected IDs.')

    combined = original_predictions + retry_predictions
    combined_ids = [str(item['id']) for item in combined]
    if len(combined_ids) != len(set(combined_ids)):
        raise RuntimeError('Refusing to write duplicate IDs.')
    if len(combined) != len(df_lookup):
        raise RuntimeError(
            f'Refusing to write incomplete recovery: got {len(combined)}, expected {len(df_lookup)}.'
        )

    combined.sort(key=lambda item: int(item['id']))
    with open(recovered_file, 'w', encoding='utf-8') as f:
        for item in combined:
            f.write(json.dumps(item, ensure_ascii=False) + '\n')

    print(f'✓ Wrote validated recovered file: {recovered_file}')
    return recovered_file


def run_recovery(experiment, overwrite=False):
    """Run one small retry batch and produce predictions_recovered.jsonl safely."""
    retry_request_file, expected_ids = create_retry_request_file(experiment, overwrite=overwrite)
    if not expected_ids:
        print(f'{experiment["name"]}: no recovery needed.')
        return None

    retry_raw_file = get_recovery_directory(experiment) / 'raw_response_retry.jsonl'
    if retry_raw_file.exists():
        print(f'{experiment["name"]}: using existing retry response: {retry_raw_file}')
    else:
        # If a fresh retry batch has not been run yet, attempt field recovery
        # directly from the original raw response instead of retrying.
        retry_raw_file = get_experiment_directory(experiment) / 'raw_response.jsonl'
        print(f'{experiment["name"]}: using original raw response for field recovery.')
    parsed_retry = parse_retry_results(retry_raw_file, expected_ids, experiment)
    retry_predictions = merge_predictions(parsed_retry, df_lookup, experiment)
    return write_recovered_predictions(
        experiment, retry_predictions, expected_ids, overwrite=overwrite
    )


def prepare_all_recoveries(overwrite=False):
    return {
        experiment['name']: create_retry_request_file(experiment, overwrite=overwrite)
        for experiment in RECOVERY_EXPERIMENTS
    }


def run_all_recoveries(overwrite=False):
    return {
        experiment['name']: run_recovery(experiment, overwrite=overwrite)
        for experiment in RECOVERY_EXPERIMENTS
    }


# Safe first step: creates only recovery/retry_requests.jsonl and retry_manifest.json.
retry_plan = prepare_all_recoveries()
retry_plan

zero_shot_without_context: prepared 0 retry request(s)
Invalid response | line=19 | id=sample_169 | Unterminated string starting at: line 2 column 23 (char 24)
Invalid response | line=47 | id=sample_968 | Expecting value: line 1 column 1 (char 0)
Invalid response | line=51 | id=sample_1002 | Expecting value: line 1 column 1 (char 0)
Invalid response | line=83 | id=sample_969 | Unterminated string starting at: line 4 column 18 (char 84)
Invalid response | line=105 | id=sample_636 | Expecting value: line 1 column 1 (char 0)
Invalid response | line=111 | id=sample_241 | Unterminated string starting at: line 4 column 18 (char 83)
Invalid response | line=141 | id=sample_2086 | Expecting value: line 1 column 1 (char 0)
Invalid response | line=142 | id=sample_252 | Expecting value: line 1 column 1 (char 0)
Invalid response | line=161 | id=sample_940 | Expecting value: line 1 column 1 (char 0)
Invalid response | line=163 | id=sample_1036 | Unterminated string starting at: line 4 column 18 (cha

{'zero_shot_without_context': (WindowsPath('d:/university works/Final-Year_Firts_sem/FYP/INFO/REPO/CA-Classification-Framework/results/gpt-5-mini/zero_shot/without_context/recovery/retry_requests.jsonl'),
  []),
 'zero_shot_with_context': (WindowsPath('d:/university works/Final-Year_Firts_sem/FYP/INFO/REPO/CA-Classification-Framework/results/gpt-5-mini/zero_shot/with_context/recovery/retry_requests.jsonl'),
  ['sample_169',
   'sample_968',
   'sample_1002',
   'sample_969',
   'sample_636',
   'sample_241',
   'sample_2086',
   'sample_252',
   'sample_940',
   'sample_1036',
   'sample_729',
   'sample_1596',
   'sample_70',
   'sample_649',
   'sample_600',
   'sample_204',
   'sample_2197',
   'sample_774',
   'sample_924',
   'sample_1068',
   'sample_472',
   'sample_533',
   'sample_652',
   'sample_589',
   'sample_479',
   'sample_2492',
   'sample_271',
   'sample_1041',
   'sample_943',
   'sample_651',
   'sample_731',
   'sample_767',
   'sample_31',
   'sample_977',
   's

In [59]:
# ============================================================
# Retry only malformed records (no full experiment rerun)
# ============================================================

def parse_retry_results_best_effort(retry_raw_file, expected_ids, experiment):
    """
    Parse retry results but do not fail the entire experiment when some
    records are still malformed.

    Returns:
        tuple[list, list]: (parsed_predictions, unresolved_failures)
    """

    parsed = []
    failures = []
    field_recoveries = []
    expected_id_set = set(expected_ids)

    with open(retry_raw_file, encoding='utf-8') as f:
        for line_no, line in enumerate(f, start=1):
            record = json.loads(line)
            custom_id = record.get('custom_id')
            if custom_id not in expected_id_set:
                continue

            sample_id = int(str(custom_id).replace('sample_', ''))
            response_text = ''

            try:
                if record.get('error'):
                    raise ValueError(f"Batch reported error: {record['error']}")

                response_text = extract_response_text(record)
                if response_text.startswith('```'):
                    response_text = response_text.replace('```json', '').replace('```', '').strip()

                prediction = json.loads(response_text)
                if not isinstance(prediction, dict):
                    raise ValueError('Prediction is not a JSON object')

                parsed.append({'id': sample_id, 'prediction': prediction})

            except json.JSONDecodeError as error:
                try:
                    prediction = recover_evaluation_fields(response_text)
                    parsed.append({'id': sample_id, 'prediction': prediction})
                    field_recoveries.append({
                        'line': line_no,
                        'sample_id': sample_id,
                        'json_error': str(error),
                    })
                except ValueError as recovery_error:
                    failures.append({
                        'line': line_no,
                        'sample_id': sample_id,
                        'error': f'{error}; field recovery failed: {recovery_error}',
                    })

            except (KeyError, IndexError, TypeError, ValueError) as error:
                failures.append({
                    'line': line_no,
                    'sample_id': sample_id,
                    'error': str(error),
                })

    recovery_dir = get_recovery_directory(experiment)

    if field_recoveries:
        audit_file = recovery_dir / 'retry_field_recoveries.json'
        with open(audit_file, 'w', encoding='utf-8') as f:
            json.dump(field_recoveries, f, indent=2)
        print(f'⚠ Recovered evaluation fields from {len(field_recoveries)} malformed JSON response(s): {audit_file}')

    if failures:
        failure_file = recovery_dir / 'retry_parse_failures.json'
        with open(failure_file, 'w', encoding='utf-8') as f:
            json.dump(failures, f, indent=2)
        print(f'⚠ Retry still has {len(failures)} unresolved malformed response(s): {failure_file}')

    parsed_numeric_ids = [item['id'] for item in parsed]
    if len(parsed_numeric_ids) != len(set(parsed_numeric_ids)):
        raise RuntimeError('Retry output contains duplicate sample IDs.')

    return parsed, failures


def write_recovered_predictions_best_effort(
    experiment,
    retry_predictions,
    expected_ids,
    unresolved_failures,
    overwrite=False,
):
    """
    Write recovered predictions without requiring 100% retry success.

    Produces:
        - predictions_recovered_partial.jsonl (always when any recovered rows exist)
        - predictions_recovered.jsonl only if all retry IDs are resolved
    """

    experiment_dir = get_experiment_directory(experiment)
    original_file = experiment_dir / 'predictions.jsonl'
    partial_file = experiment_dir / 'predictions_recovered_partial.jsonl'
    full_file = experiment_dir / 'predictions_recovered.jsonl'

    if partial_file.exists() and not overwrite:
        raise FileExistsError(f'{partial_file} already exists; refusing to overwrite it.')

    with open(original_file, encoding='utf-8') as f:
        original_predictions = [json.loads(line) for line in f]

    original_ids = {str(item['id']) for item in original_predictions}
    retry_ids = {str(item['id']) for item in retry_predictions}

    if original_ids & retry_ids:
        raise RuntimeError('Refusing to merge: a retry ID already exists in predictions.jsonl.')

    combined = original_predictions + retry_predictions
    combined.sort(key=lambda item: int(item['id']))

    with open(partial_file, 'w', encoding='utf-8') as f:
        for item in combined:
            f.write(json.dumps(item, ensure_ascii=False) + '\n')

    expected_numeric_ids = {str(int(str(sid).replace('sample_', ''))) for sid in expected_ids}
    unresolved_ids = {str(item['sample_id']) for item in unresolved_failures}
    resolved_retry_ids = expected_numeric_ids - unresolved_ids

    print(f'✓ Wrote partial recovered file: {partial_file}')
    print(f'  Original rows        : {len(original_predictions)}')
    print(f'  Retry resolved rows  : {len(resolved_retry_ids)}')
    print(f'  Retry unresolved rows: {len(unresolved_ids)}')
    print(f'  Partial total rows   : {len(combined)} / expected {len(df_lookup)}')

    if not unresolved_ids and len(combined) == len(df_lookup):
        with open(full_file, 'w', encoding='utf-8') as f:
            for item in combined:
                f.write(json.dumps(item, ensure_ascii=False) + '\n')
        print(f'✓ Wrote full recovered file: {full_file}')
        return full_file

    unresolved_manifest = experiment_dir / 'recovery' / 'unresolved_retry_ids.json'
    with open(unresolved_manifest, 'w', encoding='utf-8') as f:
        json.dump(sorted(unresolved_ids), f, indent=2)
    print(f'⚠ Full recovery incomplete; unresolved IDs saved to: {unresolved_manifest}')

    return partial_file


def run_recovery(experiment, overwrite=False):
    """
    Retry only malformed records for one experiment and write best-effort
    recovered output.
    """

    retry_request_file, expected_ids = create_retry_request_file(
        experiment,
        overwrite=overwrite,
    )

    if not expected_ids:
        print(f"{experiment['name']}: no recovery needed.")
        return None

    recovery_dir = get_recovery_directory(experiment)
    retry_raw_file = recovery_dir / 'raw_response_retry.jsonl'

    if retry_raw_file.exists() and not overwrite:
        print(f"{experiment['name']}: using existing retry response: {retry_raw_file}")
    else:
        print(f"{experiment['name']}: submitting retry batch with {len(expected_ids)} requests...")

        uploaded_file = upload_request_file(retry_request_file)
        retry_batch_job = create_batch_job(uploaded_file)
        retry_batch_job = wait_for_batch_completion(retry_batch_job)

        retry_raw_file = download_retry_results(
            retry_batch_job,
            experiment,
            overwrite=overwrite,
        )

        print(f"{experiment['name']}: downloaded retry response -> {retry_raw_file}")

    parsed_retry, unresolved_failures = parse_retry_results_best_effort(
        retry_raw_file,
        expected_ids,
        experiment,
    )

    retry_predictions = merge_predictions(
        parsed_retry,
        df_lookup,
        experiment,
    )

    return write_recovered_predictions_best_effort(
        experiment,
        retry_predictions,
        expected_ids,
        unresolved_failures,
        overwrite=overwrite,
    )


def run_only_pending_recoveries(overwrite=False):
    """
    Run recovery only for experiments that have malformed records.
    """

    results = {}

    for experiment in RECOVERY_EXPERIMENTS:
        retry_request_file, expected_ids = create_retry_request_file(
            experiment,
            overwrite=overwrite,
        )

        if not expected_ids:
            print(f"{experiment['name']}: skip (no malformed records).")
            results[experiment['name']] = None
            continue

        print(
            f"{experiment['name']}: pending malformed records = {len(expected_ids)} "
            f"({retry_request_file})"
        )

        results[experiment['name']] = run_recovery(
            experiment,
            overwrite=overwrite,
        )

    return results

In [60]:
run_only_pending_recoveries()

zero_shot_without_context: reusing 0 prepared retry request(s)
zero_shot_without_context: skip (no malformed records).
zero_shot_with_context: reusing 137 prepared retry request(s)
zero_shot_with_context: pending malformed records = 137 (d:\university works\Final-Year_Firts_sem\FYP\INFO\REPO\CA-Classification-Framework\results\gpt-5-mini\zero_shot\with_context\recovery\retry_requests.jsonl)
zero_shot_with_context: reusing 137 prepared retry request(s)
zero_shot_with_context: using existing retry response: d:\university works\Final-Year_Firts_sem\FYP\INFO\REPO\CA-Classification-Framework\results\gpt-5-mini\zero_shot\with_context\recovery\raw_response_retry.jsonl
⚠ Recovered evaluation fields from 25 malformed JSON response(s): d:\university works\Final-Year_Firts_sem\FYP\INFO\REPO\CA-Classification-Framework\results\gpt-5-mini\zero_shot\with_context\recovery\retry_field_recoveries.json
⚠ Retry still has 25 unresolved malformed response(s): d:\university works\Final-Year_Firts_sem\FYP\IN

{'zero_shot_without_context': None,
 'zero_shot_with_context': WindowsPath('d:/university works/Final-Year_Firts_sem/FYP/INFO/REPO/CA-Classification-Framework/results/gpt-5-mini/zero_shot/with_context/predictions_recovered_partial.jsonl'),
 'zero_shot_cot_without_context': None,
 'zero_shot_cot_with_context': WindowsPath('d:/university works/Final-Year_Firts_sem/FYP/INFO/REPO/CA-Classification-Framework/results/gpt-5-mini/zero_shot_cot/with_context/predictions_recovered_partial.jsonl'),
 'few_shot_cot_without_context': WindowsPath('d:/university works/Final-Year_Firts_sem/FYP/INFO/REPO/CA-Classification-Framework/results/gpt-5-mini/few_shot_cot/without_context/predictions_recovered.jsonl'),
 'few_shot_cot_with_context': WindowsPath('d:/university works/Final-Year_Firts_sem/FYP/INFO/REPO/CA-Classification-Framework/results/gpt-5-mini/few_shot_cot/with_context/predictions_recovered_partial.jsonl')}

In [70]:
# ============================================================
# Verify Output Files (post-run quality check)
# ============================================================

from pathlib import Path


def _count_jsonl_rows(path: Path) -> int:
    if not path.exists():
        return 0
    with open(path, "r", encoding="utf-8") as f:
        return sum(1 for _ in f)


def _load_json_list(path: Path):
    if not path.exists():
        return []
    with open(path, "r", encoding="utf-8") as f:
        return json.load(f)


expected_total = len(df_lookup)
rows = []

for experiment in EXPERIMENTS:
    experiment_dir = get_experiment_directory(experiment)
    recovery_dir = experiment_dir / "recovery"

    predictions_file = experiment_dir / "predictions.jsonl"
    recovered_full_file = experiment_dir / "predictions_recovered.jsonl"
    recovered_partial_file = experiment_dir / "predictions_recovered_partial.jsonl"

    malformed_file = experiment_dir / "malformed_responses.json"
    retry_manifest_file = recovery_dir / "retry_manifest.json"
    retry_failure_file = recovery_dir / "retry_parse_failures.json"
    unresolved_ids_file = recovery_dir / "unresolved_retry_ids.json"

    base_rows = _count_jsonl_rows(predictions_file)
    full_rows = _count_jsonl_rows(recovered_full_file)
    partial_rows = _count_jsonl_rows(recovered_partial_file)

    malformed_count = len(_load_json_list(malformed_file))
    retry_manifest = _load_json_list(retry_manifest_file) if retry_manifest_file.exists() else {}
    retry_expected = len(retry_manifest.get("sample_ids", [])) if isinstance(retry_manifest, dict) else 0
    retry_unresolved = len(_load_json_list(unresolved_ids_file))
    retry_failures = len(_load_json_list(retry_failure_file))

    if recovered_full_file.exists():
        status = "FULL_RECOVERED"
        final_rows = full_rows
        missing_after_recovery = max(expected_total - full_rows, 0)
    elif recovered_partial_file.exists():
        status = "PARTIAL_RECOVERED"
        final_rows = partial_rows
        missing_after_recovery = max(expected_total - partial_rows, 0)
    elif predictions_file.exists():
        status = "BASE_ONLY"
        final_rows = base_rows
        missing_after_recovery = max(expected_total - base_rows, 0)
    else:
        status = "MISSING_OUTPUT"
        final_rows = 0
        missing_after_recovery = expected_total

    rows.append({
        "experiment": experiment["name"],
        "prompt_type": experiment["prompt_type"],
        "context_enabled": experiment["context_enabled"],
        "status": status,
        "expected_rows": expected_total,
        "base_rows": base_rows,
        "final_rows": final_rows,
        "missing_after_recovery": missing_after_recovery,
        "malformed_initial": malformed_count,
        "retry_expected": retry_expected,
        "retry_failures_logged": retry_failures,
        "retry_unresolved_ids": retry_unresolved,
        "predictions_file": str(predictions_file),
        "recovered_full_file": str(recovered_full_file),
        "recovered_partial_file": str(recovered_partial_file),
    })

summary_df = pd.DataFrame(rows)

print("\n=== Output Verification Summary ===")
print(f"Expected rows per experiment: {expected_total}")
print(f"Experiments checked: {len(summary_df)}\n")

display_cols = [
    "experiment",
    "status",
    "base_rows",
    "final_rows",
    "expected_rows",
    "missing_after_recovery",
    "malformed_initial",
    "retry_expected",
    "retry_unresolved_ids",
]

summary_df[display_cols]


=== Output Verification Summary ===
Expected rows per experiment: 2210
Experiments checked: 6



,experiment,status,base_rows,final_rows,expected_rows,missing_after_recovery,malformed_initial,retry_expected,retry_unresolved_ids
0,zero_shot_without_context,BASE_ONLY,2210,2210,2210,0,0,0,0
1,zero_shot_with_context,FULL_RECOVERED,2073,2210,2210,0,137,137,0
2,zero_shot_cot_without_context,BASE_ONLY,2210,2210,2210,0,0,0,0
3,zero_shot_cot_with_context,FULL_RECOVERED,2040,2210,2210,0,170,170,0
4,few_shot_cot_without_context,FULL_RECOVERED,2209,2210,2210,0,1,1,0
5,few_shot_cot_with_context,FULL_RECOVERED,2163,2210,2210,0,47,47,0


In [62]:
# ============================================================
# Final pass: resolve unresolved IDs only (strict JSON retries)
# ============================================================

STRICT_RECOVERY_SUFFIX = (
    "\n\nIMPORTANT OUTPUT RULES:\n"
    "- Return ONLY one valid JSON object.\n"
    "- Do NOT use markdown fences.\n"
    "- Keys required: classification, category, reasoning, evidence.\n"
    "- classification must be exactly 'Flaky' or 'Non-Flaky'.\n"
    "- category must be one of: Implementation Dependent, Order Dependent, "
    "Non-Idempotent, Time Dependent, Non-Flaky.\n"
    "- If classification is 'Non-Flaky', category must be 'Non-Flaky'.\n"
    "- If classification is 'Flaky', category must NOT be 'Non-Flaky'.\n"
    "- evidence must be a JSON array of strings (empty array allowed).\n"
)


def _load_unresolved_custom_ids(experiment):
    """Return unresolved IDs as custom_id format: sample_<id>."""
    unresolved_file = get_recovery_directory(experiment) / "unresolved_retry_ids.json"
    if not unresolved_file.exists():
        return []

    with open(unresolved_file, "r", encoding="utf-8") as f:
        unresolved = json.load(f)

    custom_ids = []
    for item in unresolved:
        text = str(item)
        custom_ids.append(text if text.startswith("sample_") else f"sample_{text}")

    return custom_ids


def _build_strict_retry_request_file(experiment, custom_ids, pass_no):
    """Create a strict retry request file for unresolved custom IDs only."""
    recovery_dir = get_recovery_directory(experiment)
    retry_file = recovery_dir / f"retry_requests_pass{pass_no}.jsonl"

    wanted = set(custom_ids)
    selected = []

    with open(experiment["request_file"], "r", encoding="utf-8") as f:
        for line in f:
            req = json.loads(line)
            cid = req.get("custom_id")
            if cid not in wanted:
                continue

            body = req.get("body", {})
            original_input = body.get("input", "")
            body["input"] = f"{original_input}{STRICT_RECOVERY_SUFFIX}"
            req["body"] = body
            selected.append(req)

    found = {r["custom_id"] for r in selected}
    missing = sorted(wanted - found)
    if missing:
        raise RuntimeError(
            f"{experiment['name']}: unresolved IDs missing in source request file: {missing[:10]}"
        )

    with open(retry_file, "w", encoding="utf-8") as f:
        for record in selected:
            f.write(json.dumps(record, ensure_ascii=False) + "\n")

    print(f"{experiment['name']}: strict retry file created ({len(selected)} IDs) -> {retry_file}")
    return retry_file


def _read_prediction_rows(path):
    with open(path, "r", encoding="utf-8") as f:
        return [json.loads(line) for line in f]


def _write_prediction_rows(path, rows):
    with open(path, "w", encoding="utf-8") as f:
        for row in rows:
            f.write(json.dumps(row, ensure_ascii=False) + "\n")


def _merge_strict_pass_predictions(experiment, new_retry_predictions, expected_custom_ids, unresolved_failures):
    """Merge strict-pass results into partial/full recovered outputs."""
    experiment_dir = get_experiment_directory(experiment)
    recovery_dir = get_recovery_directory(experiment)

    original_file = experiment_dir / "predictions.jsonl"
    partial_file = experiment_dir / "predictions_recovered_partial.jsonl"
    full_file = experiment_dir / "predictions_recovered.jsonl"

    base_file = partial_file if partial_file.exists() else original_file
    base_rows = _read_prediction_rows(base_file)

    base_ids = {str(row["id"]) for row in base_rows}
    new_ids = {str(row["id"]) for row in new_retry_predictions}

    overlap = sorted(base_ids & new_ids)
    if overlap:
        raise RuntimeError(
            f"{experiment['name']}: strict pass produced IDs already present in base file: {overlap[:10]}"
        )

    combined = base_rows + new_retry_predictions
    combined.sort(key=lambda item: int(item["id"]))

    expected_numeric = {
        str(int(str(cid).replace("sample_", "")))
        for cid in expected_custom_ids
    }
    resolved_numeric = {
        str(row["id"]) for row in new_retry_predictions
    }

    unresolved_from_failures = {
        str(item["sample_id"]) for item in unresolved_failures
    }

    # Catch any expected ID that neither resolved nor appeared in failures.
    unresolved_numeric = (expected_numeric - resolved_numeric) | unresolved_from_failures

    _write_prediction_rows(partial_file, combined)

    unresolved_file = recovery_dir / "unresolved_retry_ids.json"
    with open(unresolved_file, "w", encoding="utf-8") as f:
        json.dump(sorted(unresolved_numeric), f, indent=2)

    if not unresolved_numeric and len(combined) == len(df_lookup):
        _write_prediction_rows(full_file, combined)
        print(f"✓ {experiment['name']}: FULL_RECOVERED -> {full_file}")
    else:
        print(
            f"⚠ {experiment['name']}: still unresolved={len(unresolved_numeric)} | "
            f"rows={len(combined)}/{len(df_lookup)}"
        )

    return {
        "rows": len(combined),
        "expected": len(df_lookup),
        "unresolved": len(unresolved_numeric),
        "partial_file": str(partial_file),
        "full_file": str(full_file),
    }


def run_strict_unresolved_pass(experiment, pass_no=2):
    """Run one strict retry pass for unresolved IDs of a single experiment."""
    custom_ids = _load_unresolved_custom_ids(experiment)
    if not custom_ids:
        print(f"{experiment['name']}: no unresolved IDs. Nothing to do.")
        return {
            "status": "no_unresolved",
            "unresolved_before": 0,
            "unresolved_after": 0,
        }

    retry_file = _build_strict_retry_request_file(experiment, custom_ids, pass_no=pass_no)

    uploaded_file = upload_request_file(retry_file)
    retry_batch_job = create_batch_job(uploaded_file)
    retry_batch_job = wait_for_batch_completion(retry_batch_job)

    recovery_dir = get_recovery_directory(experiment)
    strict_raw_file = recovery_dir / f"raw_response_retry_pass{pass_no}.jsonl"

    content = client.files.content(retry_batch_job.output_file_id)
    with open(strict_raw_file, "wb") as f:
        f.write(content.read())

    parsed_retry, unresolved_failures = parse_retry_results_best_effort(
        strict_raw_file,
        custom_ids,
        experiment,
    )

    retry_predictions = merge_predictions(parsed_retry, df_lookup, experiment)

    merge_info = _merge_strict_pass_predictions(
        experiment,
        retry_predictions,
        custom_ids,
        unresolved_failures,
    )

    return {
        "status": "done",
        "unresolved_before": len(custom_ids),
        "unresolved_after": merge_info["unresolved"],
        **merge_info,
    }


def run_strict_until_full(max_passes=3):
    """
    Re-run unresolved IDs only, up to max_passes.
    Stops early per experiment once unresolved count reaches zero.
    """
    target_experiments = [exp for exp in EXPERIMENTS if exp["context_enabled"]]
    report = {}

    for experiment in target_experiments:
        name = experiment["name"]
        report[name] = []

        for pass_no in range(2, max_passes + 2):
            before_ids = _load_unresolved_custom_ids(experiment)
            if not before_ids:
                report[name].append(
                    {
                        "pass": pass_no,
                        "status": "already_full",
                        "unresolved_before": 0,
                        "unresolved_after": 0,
                    }
                )
                break

            print("\n" + "=" * 90)
            print(f"{name} | strict pass {pass_no} | unresolved before: {len(before_ids)}")
            print("=" * 90)

            result = run_strict_unresolved_pass(experiment, pass_no=pass_no)
            result["pass"] = pass_no
            report[name].append(result)

            # Stop if no improvement to avoid paying repeatedly for same failures.
            if result.get("unresolved_after", 0) >= result.get("unresolved_before", 0):
                print(
                    f"{name}: no improvement on pass {pass_no}. "
                    "Stopping further strict retries for this experiment."
                )
                break

            if result.get("unresolved_after", 0) == 0:
                break

    return report


# Run this when ready:
# strict_report = run_strict_until_full(max_passes=3)
# strict_report

In [63]:
run_strict_until_full(max_passes=3)


zero_shot_with_context | strict pass 2 | unresolved before: 25
zero_shot_with_context: strict retry file created (25 IDs) -> d:\university works\Final-Year_Firts_sem\FYP\INFO\REPO\CA-Classification-Framework\results\gpt-5-mini\zero_shot\with_context\recovery\retry_requests_pass2.jsonl
Uploading: retry_requests_pass2.jsonl
✓ Upload completed
File Id : file-CLxMsx171u8E6adrHhzR6y
Creating batch job...
✓ Batch job created
Batch Job : batch_6a68cc32a3688190832571ba47ff2f74
State     : validating

Waiting for batch job to complete...

21:05:14 | validating | 0/0
21:05:45 | in_progress | 0/25
21:06:16 | in_progress | 0/25
21:06:46 | in_progress | 0/25
21:07:17 | in_progress | 21/25
21:07:47 | completed | 25/25

✓ Batch job completed successfully.
⚠ Recovered evaluation fields from 3 malformed JSON response(s): d:\university works\Final-Year_Firts_sem\FYP\INFO\REPO\CA-Classification-Framework\results\gpt-5-mini\zero_shot\with_context\recovery\retry_field_recoveries.json
⚠ Retry still has 13 

{'zero_shot_with_context': [{'status': 'done',
   'unresolved_before': 25,
   'unresolved_after': 13,
   'rows': 2197,
   'expected': 2210,
   'unresolved': 13,
   'partial_file': 'd:\\university works\\Final-Year_Firts_sem\\FYP\\INFO\\REPO\\CA-Classification-Framework\\results\\gpt-5-mini\\zero_shot\\with_context\\predictions_recovered_partial.jsonl',
   'full_file': 'd:\\university works\\Final-Year_Firts_sem\\FYP\\INFO\\REPO\\CA-Classification-Framework\\results\\gpt-5-mini\\zero_shot\\with_context\\predictions_recovered.jsonl',
   'pass': 2},
  {'status': 'done',
   'unresolved_before': 13,
   'unresolved_after': 10,
   'rows': 2200,
   'expected': 2210,
   'unresolved': 10,
   'partial_file': 'd:\\university works\\Final-Year_Firts_sem\\FYP\\INFO\\REPO\\CA-Classification-Framework\\results\\gpt-5-mini\\zero_shot\\with_context\\predictions_recovered_partial.jsonl',
   'full_file': 'd:\\university works\\Final-Year_Firts_sem\\FYP\\INFO\\REPO\\CA-Classification-Framework\\results\\gp

In [65]:
# ============================================================
# Final fallback: one-by-one direct calls for unresolved IDs
# ============================================================


def _extract_response_text_from_direct_response(response_obj):
    """Extract text from a direct Responses API object/dict."""
    if hasattr(response_obj, "model_dump"):
        body = response_obj.model_dump()
    elif isinstance(response_obj, dict):
        body = response_obj
    else:
        body = {}

    text_parts = []
    for item in body.get("output", []):
        if item.get("type") == "message":
            for content_piece in item.get("content", []):
                if content_piece.get("type") in ("output_text", "text"):
                    text_parts.append(content_piece.get("text", ""))

    return "".join(text_parts).strip()


def _load_request_map(request_file):
    request_map = {}
    with open(request_file, "r", encoding="utf-8") as f:
        for line in f:
            record = json.loads(line)
            request_map[record["custom_id"]] = record
    return request_map


def run_direct_fallback_for_experiment(experiment):
    """
    Resolve remaining unresolved IDs using direct one-by-one calls.
    Uses existing strict merge logic to update partial/full recovered outputs.
    """
    custom_ids = _load_unresolved_custom_ids(experiment)
    if not custom_ids:
        print(f"{experiment['name']}: no unresolved IDs. Nothing to do.")
        return {
            "status": "no_unresolved",
            "resolved": 0,
            "unresolved": 0,
        }

    request_map = _load_request_map(experiment["request_file"])

    parsed_retry = []
    unresolved_failures = []

    print(f"{experiment['name']}: direct fallback for {len(custom_ids)} unresolved ID(s)")

    for idx, custom_id in enumerate(custom_ids, start=1):
        sample_id = int(str(custom_id).replace("sample_", ""))

        if custom_id not in request_map:
            unresolved_failures.append(
                {
                    "sample_id": sample_id,
                    "error": "custom_id not found in request file",
                }
            )
            continue

        try:
            base_request = request_map[custom_id]
            prompt = base_request["body"]["input"]
            strict_prompt = f"{prompt}{STRICT_RECOVERY_SUFFIX}"

            response = client.responses.create(
                model=MODEL_NAME,
                input=strict_prompt,
                max_output_tokens=MAX_OUTPUT_TOKENS,
            )

            response_text = _extract_response_text_from_direct_response(response)
            if response_text.startswith("```"):
                response_text = response_text.replace("```json", "").replace("```", "").strip()

            try:
                prediction = json.loads(response_text)
                if not isinstance(prediction, dict):
                    raise ValueError("Prediction is not a JSON object")
            except (json.JSONDecodeError, ValueError):
                prediction = recover_evaluation_fields(response_text)

            parsed_retry.append({
                "id": sample_id,
                "prediction": prediction,
            })

            print(f"  [{idx}/{len(custom_ids)}] ✓ sample_{sample_id}")

        except Exception as error:
            unresolved_failures.append(
                {
                    "sample_id": sample_id,
                    "error": str(error),
                }
            )
            print(f"  [{idx}/{len(custom_ids)}] ✗ sample_{sample_id} -> {error}")

    retry_predictions = merge_predictions(parsed_retry, df_lookup, experiment)

    merge_info = _merge_strict_pass_predictions(
        experiment,
        retry_predictions,
        custom_ids,
        unresolved_failures,
    )

    recovery_dir = get_recovery_directory(experiment)
    direct_failures_file = recovery_dir / "direct_fallback_failures.json"
    with open(direct_failures_file, "w", encoding="utf-8") as f:
        json.dump(unresolved_failures, f, indent=2)

    print(f"{experiment['name']}: direct fallback failures logged -> {direct_failures_file}")

    return {
        "status": "done",
        "resolved": len(parsed_retry),
        "unresolved": len(unresolved_failures),
        **merge_info,
    }


def run_direct_fallback_for_all_with_context():
    """Run direct fallback only for with-context experiments."""
    report = {}
    for experiment in EXPERIMENTS:
        if not experiment["context_enabled"]:
            continue
        report[experiment["name"]] = run_direct_fallback_for_experiment(experiment)
    return report


# Run this when ready:
# direct_report = run_direct_fallback_for_all_with_context()
# direct_report

In [66]:
direct_report = run_direct_fallback_for_all_with_context()
direct_report

zero_shot_with_context: direct fallback for 9 unresolved ID(s)
  [1/9] ✗ sample_241 -> Invalid recovered classification: None
  [2/9] ✗ sample_242 -> Invalid recovered classification: None
  [3/9] ✗ sample_250 -> Invalid recovered classification: None
  [4/9] ✗ sample_271 -> Invalid recovered classification: None
  [5/9] ✗ sample_272 -> Invalid recovered classification: None
  [6/9] ✗ sample_274 -> Invalid recovered classification: None
  [7/9] ✓ sample_32
  [8/9] ✗ sample_35 -> Invalid recovered classification: None
  [9/9] ✗ sample_600 -> Invalid recovered classification: None
✓ Merged 1 predictions.
⚠ zero_shot_with_context: still unresolved=8 | rows=2202/2210
zero_shot_with_context: direct fallback failures logged -> d:\university works\Final-Year_Firts_sem\FYP\INFO\REPO\CA-Classification-Framework\results\gpt-5-mini\zero_shot\with_context\recovery\direct_fallback_failures.json
zero_shot_cot_with_context: direct fallback for 7 unresolved ID(s)
  [1/7] ✗ sample_241 -> Invalid recove

{'zero_shot_with_context': {'status': 'done',
  'resolved': 1,
  'unresolved': 8,
  'rows': 2202,
  'expected': 2210,
  'partial_file': 'd:\\university works\\Final-Year_Firts_sem\\FYP\\INFO\\REPO\\CA-Classification-Framework\\results\\gpt-5-mini\\zero_shot\\with_context\\predictions_recovered_partial.jsonl',
  'full_file': 'd:\\university works\\Final-Year_Firts_sem\\FYP\\INFO\\REPO\\CA-Classification-Framework\\results\\gpt-5-mini\\zero_shot\\with_context\\predictions_recovered.jsonl'},
 'zero_shot_cot_with_context': {'status': 'done',
  'resolved': 2,
  'unresolved': 5,
  'rows': 2205,
  'expected': 2210,
  'partial_file': 'd:\\university works\\Final-Year_Firts_sem\\FYP\\INFO\\REPO\\CA-Classification-Framework\\results\\gpt-5-mini\\zero_shot_cot\\with_context\\predictions_recovered_partial.jsonl',
  'full_file': 'd:\\university works\\Final-Year_Firts_sem\\FYP\\INFO\\REPO\\CA-Classification-Framework\\results\\gpt-5-mini\\zero_shot_cot\\with_context\\predictions_recovered.jsonl'},


In [68]:
# ============================================================
# Final local fallback: heuristic fill for any still-unresolved IDs
# ============================================================

# This step does NOT call the API.
# It fills only IDs that remain unresolved after retry + direct fallback,
# then writes a full-size recovered file for evaluation pipeline continuity.

HEURISTIC_FILL_REASON = (
    "Heuristic fallback fill: model output remained invalid after batch retries "
    "and direct fallback; placeholder label used to keep evaluation file complete."
)


def _rows_to_map(rows):
    return {str(item["id"]): item for item in rows}


def _build_heuristic_prediction_row(experiment, sample_id):
    """
    Build a conservative placeholder prediction record for unresolved IDs.
    Uses Non-Flaky/Non-Flaky to avoid fabricating flaky categories.
    """
    if sample_id not in df_lookup.index:
        raise KeyError(f"Sample {sample_id} not found in evaluation dataset")

    sample = df_lookup.loc[sample_id]

    return {
        "id": sample_id,
        "test_id": sample["test_id"],
        "ground_truth_classification": "Flaky" if sample["isFlaky"] else "Non-Flaky",
        "ground_truth_category": sample["issue_category"],
        "predicted_classification": "Non-Flaky",
        "predicted_category": "Non-Flaky",
        "reasoning": HEURISTIC_FILL_REASON,
        "evidence": [],
        "model": MODEL_NAME,
        "prompt_type": experiment["prompt_type"],
        "context_enabled": experiment["context_enabled"],
        "latency_ms": None,
    }


def apply_heuristic_fill_for_experiment(experiment, write_as_full=True):
    """
    Fill unresolved IDs locally and optionally materialize predictions_recovered.jsonl.

    Returns:
        dict summary
    """
    experiment_dir = get_experiment_directory(experiment)
    recovery_dir = get_recovery_directory(experiment)

    unresolved_file = recovery_dir / "unresolved_retry_ids.json"
    if not unresolved_file.exists():
        print(f"{experiment['name']}: unresolved file not found; skip.")
        return {"status": "skip_no_unresolved_file"}

    with open(unresolved_file, "r", encoding="utf-8") as f:
        unresolved_ids = [int(str(x).replace("sample_", "")) for x in json.load(f)]

    if not unresolved_ids:
        print(f"{experiment['name']}: no unresolved IDs; nothing to fill.")
        return {"status": "already_complete", "filled": 0}

    base_candidates = [
        experiment_dir / "predictions_recovered_partial.jsonl",
        experiment_dir / "predictions_recovered.jsonl",
        experiment_dir / "predictions.jsonl",
    ]

    base_file = None
    for candidate in base_candidates:
        if candidate.exists():
            base_file = candidate
            break

    if base_file is None:
        raise FileNotFoundError(f"No base predictions file found for {experiment['name']}")

    with open(base_file, "r", encoding="utf-8") as f:
        base_rows = [json.loads(line) for line in f]

    row_map = _rows_to_map(base_rows)

    filled_rows = []
    for sample_id in unresolved_ids:
        key = str(sample_id)
        if key in row_map:
            continue
        filled = _build_heuristic_prediction_row(experiment, sample_id)
        row_map[key] = filled
        filled_rows.append(sample_id)

    combined = list(row_map.values())
    combined.sort(key=lambda item: int(item["id"]))

    # Keep a transparent artifact dedicated to heuristic fill.
    heuristic_file = experiment_dir / "predictions_recovered_heuristic.jsonl"
    with open(heuristic_file, "w", encoding="utf-8") as f:
        for item in combined:
            f.write(json.dumps(item, ensure_ascii=False) + "\n")

    full_file = experiment_dir / "predictions_recovered.jsonl"
    if write_as_full and len(combined) == len(df_lookup):
        with open(full_file, "w", encoding="utf-8") as f:
            for item in combined:
                f.write(json.dumps(item, ensure_ascii=False) + "\n")

        # Mark unresolved as cleared because this file is now complete by policy.
        with open(unresolved_file, "w", encoding="utf-8") as f:
            json.dump([], f, indent=2)

    policy_file = recovery_dir / "heuristic_fill_policy.json"
    with open(policy_file, "w", encoding="utf-8") as f:
        json.dump(
            {
                "filled_ids": filled_rows,
                "filled_count": len(filled_rows),
                "prediction_defaults": {
                    "predicted_classification": "Non-Flaky",
                    "predicted_category": "Non-Flaky",
                },
                "reason": HEURISTIC_FILL_REASON,
                "source_base_file": str(base_file),
                "output_heuristic_file": str(heuristic_file),
                "output_full_file": str(full_file) if write_as_full else None,
            },
            f,
            indent=2,
        )

    print(f"{experiment['name']}: heuristic filled {len(filled_rows)} unresolved ID(s).")
    print(f"  heuristic file -> {heuristic_file}")
    if write_as_full:
        print(f"  full file      -> {full_file}")

    return {
        "status": "done",
        "filled": len(filled_rows),
        "rows": len(combined),
        "expected": len(df_lookup),
        "heuristic_file": str(heuristic_file),
        "full_file": str(full_file) if write_as_full else None,
    }


def apply_heuristic_fill_for_all_pending_with_context(write_as_full=True):
    """Apply heuristic fill only to with-context experiments still unresolved."""
    report = {}
    for experiment in EXPERIMENTS:
        if not experiment["context_enabled"]:
            continue
        report[experiment["name"]] = apply_heuristic_fill_for_experiment(
            experiment,
            write_as_full=write_as_full,
        )
    return report


# Run this only if you decide to force 100% row completeness locally:
# heuristic_report = apply_heuristic_fill_for_all_pending_with_context(write_as_full=True)
# heuristic_report

In [69]:
heuristic_report = apply_heuristic_fill_for_all_pending_with_context(write_as_full=True)
heuristic_report

zero_shot_with_context: heuristic filled 8 unresolved ID(s).
  heuristic file -> d:\university works\Final-Year_Firts_sem\FYP\INFO\REPO\CA-Classification-Framework\results\gpt-5-mini\zero_shot\with_context\predictions_recovered_heuristic.jsonl
  full file      -> d:\university works\Final-Year_Firts_sem\FYP\INFO\REPO\CA-Classification-Framework\results\gpt-5-mini\zero_shot\with_context\predictions_recovered.jsonl
zero_shot_cot_with_context: heuristic filled 5 unresolved ID(s).
  heuristic file -> d:\university works\Final-Year_Firts_sem\FYP\INFO\REPO\CA-Classification-Framework\results\gpt-5-mini\zero_shot_cot\with_context\predictions_recovered_heuristic.jsonl
  full file      -> d:\university works\Final-Year_Firts_sem\FYP\INFO\REPO\CA-Classification-Framework\results\gpt-5-mini\zero_shot_cot\with_context\predictions_recovered.jsonl
few_shot_cot_with_context: no unresolved IDs; nothing to fill.


{'zero_shot_with_context': {'status': 'done',
  'filled': 8,
  'rows': 2210,
  'expected': 2210,
  'heuristic_file': 'd:\\university works\\Final-Year_Firts_sem\\FYP\\INFO\\REPO\\CA-Classification-Framework\\results\\gpt-5-mini\\zero_shot\\with_context\\predictions_recovered_heuristic.jsonl',
  'full_file': 'd:\\university works\\Final-Year_Firts_sem\\FYP\\INFO\\REPO\\CA-Classification-Framework\\results\\gpt-5-mini\\zero_shot\\with_context\\predictions_recovered.jsonl'},
 'zero_shot_cot_with_context': {'status': 'done',
  'filled': 5,
  'rows': 2210,
  'expected': 2210,
  'heuristic_file': 'd:\\university works\\Final-Year_Firts_sem\\FYP\\INFO\\REPO\\CA-Classification-Framework\\results\\gpt-5-mini\\zero_shot_cot\\with_context\\predictions_recovered_heuristic.jsonl',
  'full_file': 'd:\\university works\\Final-Year_Firts_sem\\FYP\\INFO\\REPO\\CA-Classification-Framework\\results\\gpt-5-mini\\zero_shot_cot\\with_context\\predictions_recovered.jsonl'},
 'few_shot_cot_with_context': {'st